# Week 2 — Preprocess images + baseline Tesseract gap analysis

This notebook replaces the handout's grayscale -> squash-resize -> hard-threshold
pipeline with `src/preprocessing.py`'s deskew + adaptive-contrast + letterbox
pipeline (see that file's docstring for *why* — short version: a fixed-size
squash and a global binary threshold both destroy information a transformer
OCR model needs). It then runs the same Tesseract baseline the handout asks
for, but scores it with CER/WER instead of eyeballing 5 outputs.


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os

REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    # Replace with your own fork's URL if you're running this outside the
    # original solution repo.
    !git clone https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- pulling latest instead of re-cloning.')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

# --upgrade matters: without it, pip can leave an already-installed
# transformers version in place instead of honouring requirements.txt's pin.
!pip install -q --upgrade -r requirements.txt

import sys
if 'src' not in sys.path:
    sys.path.insert(0, 'src')

import transformers
print('transformers version:', transformers.__version__)
assert transformers.__version__ < '5.0.0', (
    f'Got transformers {transformers.__version__} (>=5.0), which has a known '
    'TrOCR tokenizer-loading bug. requirements.txt pins <5.0.0 -- if you '
    'still see this, run `Runtime > Restart session` and re-run this cell.'
)
print('Setup complete.')


## Part A — Preprocess every collected image

In [ ]:
from preprocessing import preprocess_folder, PreprocessConfig

count = preprocess_folder('data/raw', 'data/processed', PreprocessConfig())
print(f'Preprocessed {count} images -> data/processed/')


In [ ]:
# Visual sanity check: compare a few raw vs. processed images.
import glob
from PIL import Image
import matplotlib.pyplot as plt

raw_samples = sorted(glob.glob('data/raw/**/*.png', recursive=True))[:3] + \
              sorted(glob.glob('data/raw/**/*.jpg', recursive=True))[:3]
fig, axes = plt.subplots(len(raw_samples[:3]), 2, figsize=(10, 3 * len(raw_samples[:3])))
for i, raw_path in enumerate(raw_samples[:3]):
    import os
    processed_path = os.path.join('data/processed', os.path.basename(raw_path))
    axes[i, 0].imshow(Image.open(raw_path)); axes[i, 0].set_title('raw'); axes[i, 0].axis('off')
    if os.path.exists(processed_path):
        axes[i, 1].imshow(Image.open(processed_path)); axes[i, 1].set_title('processed'); axes[i, 1].axis('off')
plt.tight_layout()
plt.show()


## Part B — Baseline Tesseract OCR + quantified gap analysis

Install Tesseract's Urdu language pack first (Colab shell command):

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-urd
!pip install -q pytesseract

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'scripts/tesseract_baseline.py',
     '--labels-csv', 'data/labels.csv',
     '--report-csv', 'reports/tesseract_baseline.csv'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


## Why We Need a Better Model

*(copy this section, filled in with your printed CER%/examples above, into your GitHub README)*

Tesseract fails on Urdu because... its Urdu language model was trained on a much
smaller and less varied dataset than its English one, and it does not handle
Nastaliq's diagonal letter-stacking and heavy contextual joining well — it
tends to segment the image into a flat left-to-right strip of isolated glyph
guesses, which breaks down badly whenever letters overlap vertically or a
word's letterforms change with position. The character error rate printed
above (see `reports/tesseract_baseline.csv` for every example) quantifies
just how large that gap is, which is exactly the gap Weeks 3-4's fine-tuned
model is built to close.
